In [8]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
import os
import sys
from sklearn.model_selection import train_test_split
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu" )
print(device)

cuda:3


In [9]:
class DF_HUST():
    def __init__(self,args):
        self.normalization = True
        self.normalization_method = args.normalization_method # min-max, z-score
        self.args = args

    def _3_sigma(self, Ser1):
        rule = (Ser1.mean() - 3 * Ser1.std() > Ser1) | (Ser1.mean() + 3 * Ser1.std() < Ser1)
        index = np.arange(Ser1.shape[0])[rule]
        return index

    def delete_3_sigma(self,df):
        df = df.replace([np.inf, -np.inf], np.nan)
        df = df.dropna()
        df = df.reset_index(drop=True)
        out_index = []
        for col in df.columns:
            index = self._3_sigma(df[col])
            out_index.extend(index)
        out_index = list(set(out_index))
        df = df.drop(out_index, axis=0)
        df = df.reset_index(drop=True)
        return df

    def read_one_csv(self,file_name,nominal_capacity=None):
        df = pd.read_csv(file_name)
        df.insert(df.shape[1]-1,'cycle index',np.arange(df.shape[0]))

        df = self.delete_3_sigma(df)

        if nominal_capacity is not None:
            #print(f'nominal_capacity:{nominal_capacity}, capacity max:{df["capacity"].max()}',end=',')
            df['capacity'] = df['capacity']/nominal_capacity
            #print(f'SOH max:{df["capacity"].max()}')
            f_df = df.iloc[:,:-1]
            if self.normalization_method == 'min-max':
                f_df = 2*(f_df - f_df.min())/(f_df.max() - f_df.min()) - 1
            elif self.normalization_method == 'z-score':
                f_df = (f_df - f_df.mean())/f_df.std()

            df.iloc[:,:-1] = f_df

        return df

    def load_one_battery(self,path,nominal_capacity=None):
        df = self.read_one_csv(path,nominal_capacity)
        # HUST数据特征选择
        # Var2：'CC Q','cycle index'
        # Var3：'CC Q','voltage entropy','cycle index'
        df = df.filter(items=['CC Q','voltage entropy','cycle index','capacity']) # CC Q在恒流充电时等价于CC charge time
        x = df.iloc[:,:-1].values
        y = df.iloc[:,-1].values
        x1 = x[:-1]
        x2 = x[1:]
        y1 = y[:-1]
        y2 = y[1:]
        return (x,y),(x1,y1),(x2,y2)

    def load_all_battery(self,path_list,nominal_capacity):
        X, Y, X1, X2, Y1, Y2 = [], [], [], [], [], []
        for path in path_list:
            (x, y),(x1, y1), (x2, y2) = self.load_one_battery(path, nominal_capacity)
            X.append(x)
            X1.append(x1)
            X2.append(x2)
            Y.append(y)
            Y1.append(y1)
            Y2.append(y2)

        X = np.concatenate(X, axis=0)
        X1 = np.concatenate(X1, axis=0)
        X2 = np.concatenate(X2, axis=0)
        Y = np.concatenate(Y, axis=0)
        Y1 = np.concatenate(Y1, axis=0)
        Y2 = np.concatenate(Y2, axis=0)

        tensor_X = torch.from_numpy(X).float().to(device) 
        tensor_X1 = torch.from_numpy(X1).float().to(device)
        tensor_X2 = torch.from_numpy(X2).float().to(device)
        tensor_Y = torch.from_numpy(Y).float().view(-1,1).to(device)
        tensor_Y1 = torch.from_numpy(Y1).float().view(-1,1).to(device)
        tensor_Y2 = torch.from_numpy(Y2).float().view(-1,1).to(device)

        train_X1, valid_X1, train_X2, valid_X2, train_Y1, valid_Y1, train_Y2, valid_Y2 = \
            train_test_split(tensor_X1, tensor_X2, tensor_Y1, tensor_Y2, test_size=0.2, random_state=420)
        train_loader = DataLoader(TensorDataset(train_X1, train_X2, train_Y1, train_Y2),
                                  batch_size=self.args.batch_size,
                                  shuffle=True)
        valid_loader = DataLoader(TensorDataset(valid_X1, valid_X2, valid_Y1, valid_Y2),
                                  batch_size=self.args.batch_size,
                                  shuffle=True)
        test_loader = DataLoader(TensorDataset(tensor_X1, tensor_X2, tensor_Y1, tensor_Y2),
                                 batch_size=self.args.batch_size,
                                 shuffle=False)

        data = {'input': tensor_X, 'label': tensor_Y,
                  'train_loader': train_loader,
                  'valid_loader': valid_loader,
                  'test_loader': test_loader}

        return data

In [ ]:
class HUSTdataFilter(DF_HUST):
    def __init__(self,root='../Data/HUST data',args=None):
        super(HUSTdataFilter, self).__init__(args)
        self.root = root
        if self.normalization:
            self.nominal_capacity = 1.1
        else:
            self.nominal_capacity = None

    def read_all(self,specific_path_list=None):
        if specific_path_list is None:
            file_list = []
            files = os.listdir(self.root)
            for file in files:
                path = os.path.join(self.root,file)
                file_list.append(path)
            return self.load_all_battery(path_list=file_list, nominal_capacity=self.nominal_capacity)
        else:
            return self.load_all_battery(path_list=specific_path_list, nominal_capacity=self.nominal_capacity)

In [ ]:
def load_HUST_data_filter(args,small_sample=None):   # 无差分，差分：diff
    test_id = ['1-4','1-8','2-4','2-8',
               '3-4','3-8','4-4','4-8',
               '5-4','5-7','6-4','6-8',
               '7-4','7-8','8-4','8-8',
               '9-4','9-8','10-4','10-8']
    data = HUSTdataFilter(root='../Data/HUST data',args=args)
    train_list = []
    test_list = []
    files = os.listdir('../Data/HUST data')
    for f in files:
        if f[:-4] in test_id:
            test_list.append(f'../Data/HUST data/{f}')
        else:
            train_list.append(f'../Data/HUST data/{f}')
    if small_sample is not None:
        train_list = train_list[:small_sample]

    train_data = data.read_all(specific_path_list=train_list)
    test_data = data.read_all(specific_path_list=test_list)
    
    dataset = {'train_input':train_data['input'],
                  'train_label':train_data['label'],
                  'test_input':test_data['input'],
                  'test_label':test_data['label'],
                  'train_loader':train_data['train_loader'],
                  'valid_loader':train_data['valid_loader'],
                  'test_loader':test_data['test_loader']}
    return dataset

In [12]:
import argparse
parser = argparse.ArgumentParser('存储过滤出关键健康指标的HUST数据集')
parser.add_argument('--dataset',type=str,default='HUST',choices=['XJTU','HUST','MIT','TJU'])
parser.add_argument('--data_root', type=str, default='../data/HUST data', help='HUST数据集根路径')
parser.add_argument('--normalization_method',type=str, default='min-max', help='min-max,z-score')
parser.add_argument('--batch_size',type=int,default=512)
args, _ = parser.parse_known_args()

In [13]:
dataset = load_HUST_data_filter(args)
torch.save(dataset, '../Data/HUST_Data_Var3.pt')

/tmp/ipykernel_1213677/2894827684.py:41: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0       1.000000
1       0.921569
2       0.921569
3       0.921569
4       0.921569
          ...   
1341   -0.960784
1342   -0.960784
1343   -0.921569
1344   -1.000000
1345   -1.000000
Name: CC charge time, Length: 1346, dtype: float64' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.iloc[:,:-1] = f_df
/tmp/ipykernel_1213677/2894827684.py:41: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0       0.647059
1       0.705882
2       0.705882
3       0.705882
4       0.764706
          ...   
1341   -1.000000
1342   -0.941176
1343   -0.941176
1344   -0.941176
1345   -0.952941
Name: CV charge time, Length: 1346, dtype: float64' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.i

In [14]:
import gc
torch.cuda.empty_cache()
gc.collect()

72